In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import re, math

In [2]:
# data frame
fs_df = pd.read_parquet("../data/foursquare_budapest_places.parquet")
print(fs_df.shape)

# drop POIs with no category information
fs_df = fs_df[~fs_df["fsq_category_labels"].isna()]
print(fs_df.shape)

(100421, 23)
(89190, 23)


In [3]:
# categories are provided as a dirty list in the fsq_category_labels column -- lets create a nice categories column
def parse_list(s):
    """
    creates a categories column with a clean list of categories
    AND
    counts the levels of information on the categories available for the POI (levels column)
    """
     # ---- coerce to a usable string ----
    if s is None or (isinstance(s, float) and math.isnan(s)):
        return []
    if isinstance(s, list):
        s = ", ".join(map(str, s))  # turn real lists into a comma-sep string
    elif not isinstance(s, str):
        s = str(s)

    # remove newline and carriage return characters
    s = s.replace("\n", " ").replace("\r", " ")

    # remove outer square brackets if they exist
    s = s.strip("[]")

    # insert a comma between adjacent quoted segments that have no separator.
    # this handles cases where one segment ends with a quote and the next starts with a quote.
    s = re.sub(r'([\'"])\s+(?=[\'"])', r'\1, ', s)

    # split the string by commas into segments.
    segments = re.split(r'\s*,\s*', s)

    final_list = []
    for seg in segments:
        # remove any stray leading/trailing quotes and extra spaces
        seg = seg.strip().strip('\'"')
        # normalize the separator: ensure " > " has one space on each side
        seg = re.sub(r'\s*>\s*', ' > ', seg)
        if seg:
            # now split by the normalized separator and add each level
            parts = [part.strip() for part in seg.split(" > ") if part.strip()]
            final_list.extend(parts)
    return final_list

fs_df['categories'] = fs_df['fsq_category_labels'].apply(parse_list)
fs_df['levels'] = fs_df['categories'].apply(len)

In [4]:
# drop rows with 0 levels -- no rows luckily
fs_df = fs_df[fs_df['levels']>0]

# create a top category column and a second best -- if available category
fs_df['top'] = fs_df['categories'].apply(lambda x: x[0])
fs_df['category'] = fs_df['categories'].apply(lambda x: x[1] if len(x) > 1 else x[0])

In [5]:

print("Unique top categories : ", len(fs_df['top'].unique()))
print("Unique more detailed (second or top) categories : ", len(fs_df['category'].unique()))

Unique top categories :  10
Unique more detailed (second or top) categories :  367


In [6]:
# print top
fs_df['top'].unique()

array(['Retail', 'Sports and Recreation',
       'Business and Professional Services', 'Travel and Transportation',
       'Event', 'Arts and Entertainment', 'Dining and Drinking',
       'Landmarks and Outdoors', 'Community and Government',
       'Health and Medicine'], dtype=object)

In [7]:
# print the most frequent categories in the more detailed category column
counts = fs_df['category'].value_counts()
counts.head(20)

category
Office                                7331
Restaurant                            6857
Education                             4342
Health and Beauty Service             3591
Bar                                   3228
Fashion Retail                        2597
Transport Hub                         2516
Food and Beverage Retail              2444
Construction Supplies Store           2416
Financial Service                     1935
Lodging                               1825
Automotive Service                    1752
Road                                  1737
Cafe                                  1724
Retail                                1638
Business and Professional Services    1633
Computers and Electronics Retail      1402
Gym and Studio                        1337
Government Building                   1221
Park                                  1212
Name: count, dtype: int64